# ⚛️ Atomic-1Bit — Train Flagship Instruct Model (Colab)

Train the **Flagship 12.5M** instruct model on [Alpaca Cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned).

| Param | Value |
|---|---|
| **Params** | ~12.5M |
| **Dim** | 320 |
| **Depth** | 8 |
| **Heads** | 5 |
| **Vocab** | 4096 (frequency-filtered) |
| **Context** | 256 |
| **Effective Batch** | 256 (32 × 8 grad accum) |
| **Scheduler** | Cosine with linear warmup |
| **Gradient Clipping** | 1.0 |
| **Weight Decay** | 0.1 (non-norm/bias/emb) |

**Runtime**: Select **GPU** (Runtime → Change runtime type → T4 GPU).

> ⚡ This is the largest model. A T4 GPU is sufficient but training will be significantly faster on V100/A100 (Colab Pro).

## 1 · Setup

In [ ]:
!pip install -q torch tiktoken datasets numpy matplotlib tqdm pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/Atomic-1Bit/weights'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_DIR}')

## 2 · Model Code (Inlined)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

def activation_quant(x):
    scale = 127.0 / x.abs().max(dim=-1, keepdim=True)[0].clamp(min=1e-5)
    y = (x * scale).round().clamp(-127, 127)
    y_ste = (y - x * scale).detach() + x * scale
    return y_ste, scale

def weight_quant(w):
    scale = 1.0 / w.abs().mean().clamp(min=1e-5)
    y = (w * scale).round().clamp(-1, 1)
    y_ste = (y - w * scale).detach() + w * scale
    return y_ste, scale

class BitLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)
        self.eps = 1e-5

    def forward(self, x):
        x_f32 = x.float()
        rms = torch.sqrt(torch.mean(x_f32 ** 2, dim=-1, keepdim=True) + self.eps)
        x_norm = x_f32 / rms
        x_quant_ste, scale_x = activation_quant(x_norm)
        w_quant_ste, scale_w = weight_quant(self.weight)
        y = F.linear(x_quant_ste, w_quant_ste)
        y_out = y / (scale_x * scale_w)
        if self.bias is not None:
            y_out += self.bias
        return y_out

@dataclass
class AtomicConfig:
    vocab_size: int = 50257
    dim: int = 512
    depth: int = 8
    heads: int = 8
    context_length: int = 1024

class BitAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.dim % config.heads == 0
        self.dim = config.dim
        self.heads = config.heads
        self.head_dim = config.dim // config.heads
        self.q_proj = BitLinear(config.dim, config.dim)
        self.k_proj = BitLinear(config.dim, config.dim)
        self.v_proj = BitLinear(config.dim, config.dim)
        self.o_proj = BitLinear(config.dim, config.dim)

    def forward(self, x, kv_cache=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.heads, self.head_dim).transpose(1, 2)
        if kv_cache is not None:
            cached_k, cached_v = kv_cache
            k = torch.cat([cached_k, k], dim=2)
            v = torch.cat([cached_v, v], dim=2)
        new_kv_cache = (k, v)
        T_total = k.shape[2]
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        mask = torch.ones(T, T_total, device=x.device, dtype=torch.bool)
        mask = torch.triu(mask, diagonal=T_total - T + 1)
        att = att.masked_fill(mask, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv_cache

class BitFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_dim = 4 * config.dim
        self.fc1 = BitLinear(config.dim, hidden_dim)
        self.fc2 = BitLinear(hidden_dim, config.dim)
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class AtomicBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.RMSNorm(config.dim, eps=1e-5)
        self.attn = BitAttention(config)
        self.ln2 = nn.RMSNorm(config.dim, eps=1e-5)
        self.mlp = BitFeedForward(config)

    def forward(self, x, kv_cache=None):
        attn_out, new_kv_cache = self.attn(self.ln1(x), kv_cache=kv_cache)
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x, new_kv_cache

class AtomicTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.dim)
        self.pos_emb = nn.Embedding(config.context_length, config.dim)
        self.layers = nn.ModuleList([AtomicBlock(config) for _ in range(config.depth)])
        self.ln_f = nn.RMSNorm(config.dim, eps=1e-5)
        self.head = BitLinear(config.dim, config.vocab_size)

    def forward(self, idx, kv_cache=None):
        B, T = idx.shape
        if kv_cache is not None and kv_cache[0] is not None:
            pos_offset = kv_cache[0][0].shape[2]
        else:
            pos_offset = 0
        pos = torch.arange(pos_offset, pos_offset + T, dtype=torch.long, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)
        new_kv_cache = []
        for i, layer in enumerate(self.layers):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, new_cache = layer(x, kv_cache=layer_cache)
            new_kv_cache.append(new_cache)
        x = self.ln_f(x)
        logits = self.head(x)
        if kv_cache is not None:
            return logits, new_kv_cache
        return logits


def init_weights(model):
    """Apply scaled Kaiming initialization to BitLinear latent weights."""
    for name, module in model.named_modules():
        if isinstance(module, BitLinear):
            torch.nn.init.kaiming_normal_(module.weight, a=math.sqrt(5))

print('✅ Model code loaded.')

## 3 · Dataset

In [ ]:
import json
import numpy as np
import tiktoken
import time
from collections import Counter
from datasets import load_dataset

# -------- Hyperparameters --------
BATCH_SIZE       = 32
GRAD_ACCUM_STEPS = 8       # Effective batch = 32 × 8 = 256
CONTEXT_LEN      = 256
DIM              = 320
DEPTH            = 8
HEADS            = 5
VOCAB_SIZE       = 4096
UNK_ID           = VOCAB_SIZE - 1
LR               = 3e-4
WARMUP_STEPS     = 1000
CLIP_GRAD        = 1.0
WEIGHT_DECAY     = 0.1
# ---------------------------------

class EfficientInstructDataset:
    def __init__(self, split='train', context_length=256, vocab_file=None):
        if vocab_file is None:
            vocab_file = os.path.join(DRIVE_DIR, 'vocab_map_instruct.json')
        print(f'Loading Alpaca Cleaned ({split})...')
        raw_dataset = load_dataset('yahma/alpaca-cleaned', split=split)
        self.enc = tiktoken.get_encoding('gpt2')
        self.context_length = context_length
        self.vocab_file = vocab_file

        print(f'Filtering dataset (Max Tokens: {context_length}, Min Tokens: 10)...')
        def filter_fn(sample):
            text = self.format_prompt(sample)
            ids = self.enc.encode(text)
            return 10 <= len(ids) + 1 <= context_length

        self.dataset = raw_dataset.filter(filter_fn)
        print(f'Kept {len(self.dataset)}/{len(raw_dataset)} clean samples.')

        self.token_map = {}
        self.reverse_map = {}
        self._init_vocab()

    def _init_vocab(self):
        if os.path.exists(self.vocab_file):
            print(f'Loading vocab map from {self.vocab_file}...')
            with open(self.vocab_file, 'r') as f:
                data = json.load(f)
                self.token_map = {int(k): v for k, v in data['token_map'].items()}
                self.reverse_map = {int(k): v for k, v in data['reverse_map'].items()}
            print(f'Loaded {len(self.token_map)} mapped tokens.')
            return

        print('Building Frequency-Based Vocab (Scanning first 20k filtered samples)...')
        counter = Counter()
        scan_limit = min(20000, len(self.dataset))
        for i in range(scan_limit):
            row = self.dataset[i]
            text = self.format_prompt(row)
            ids = self.enc.encode(text)
            counter.update(ids)
        eot = self.enc.eot_token
        most_common = counter.most_common(VOCAB_SIZE - 2)
        new_id = 0
        valid_gpt_ids = [k for k, v in most_common]
        if eot not in valid_gpt_ids:
            valid_gpt_ids.append(eot)
        valid_gpt_ids = valid_gpt_ids[:VOCAB_SIZE - 1]
        for gpt_id in valid_gpt_ids:
            self.token_map[gpt_id] = new_id
            self.reverse_map[new_id] = gpt_id
            new_id += 1
        self.unk_token = UNK_ID
        print(f'Saving vocab map to {self.vocab_file}...')
        os.makedirs(os.path.dirname(self.vocab_file), exist_ok=True)
        with open(self.vocab_file, 'w') as f:
            json.dump({'token_map': self.token_map, 'reverse_map': self.reverse_map}, f)

    def format_prompt(self, sample):
        text = f"### Instruction: {sample['instruction']}\n"
        if sample.get('input', ''):
            text += f"### Input: {sample['input']}\n"
        text += f"### Response: {sample['output']}"
        return text

    def get_batch(self, batch_size):
        indices = np.random.randint(0, len(self.dataset), batch_size)
        rows = self.dataset.select(indices)
        batch_input_ids, batch_targets = [], []
        for i in range(len(rows)):
            row = rows[i]
            text = self.format_prompt(row)
            gpt_ids = self.enc.encode(text)
            gpt_ids.append(self.enc.eot_token)
            pocket_ids = [self.token_map.get(gid, UNK_ID) for gid in gpt_ids]
            if len(pocket_ids) < self.context_length + 1:
                eot_mapped = self.token_map.get(self.enc.eot_token, UNK_ID)
                pocket_ids += [eot_mapped] * (self.context_length + 1 - len(pocket_ids))
            batch_input_ids.append(pocket_ids[:-1])
            batch_targets.append(pocket_ids[1:])
        return torch.tensor(batch_input_ids, dtype=torch.long), torch.tensor(batch_targets, dtype=torch.long)

print('✅ Dataset class ready.')

## 4 · Training Config

In [ ]:
ADDITIONAL_STEPS = 5000
USE_AMP = True

## 5 · Train

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import csv

def get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps, min_lr_ratio=0.01):
    """Linear warmup followed by cosine decay."""
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(min_lr_ratio, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def generate_demo(model, ds, instruction='What is AI?', max_tokens=60):
    model.eval()
    device = next(model.parameters()).device
    prompt = f"### Instruction: {instruction}\n### Response:"
    gpt_ids = ds.enc.encode(prompt)
    ids = [ds.token_map.get(gid, UNK_ID) for gid in gpt_ids]
    x = torch.tensor([ids], dtype=torch.long).to(device)
    eot_mapped = ds.token_map.get(ds.enc.eot_token, UNK_ID)
    tokens = []
    for _ in range(max_tokens):
        if x.size(1) >= CONTEXT_LEN:
            break
        with torch.no_grad():
            logits = model(x)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            next_token = torch.multinomial(probs, 1)
            pocket_id = next_token.item()
            gpt_id = ds.reverse_map.get(pocket_id, ds.enc.eot_token)
            try:
                tokens.append(ds.enc.decode([gpt_id]))
            except:
                pass
            x = torch.cat([x, next_token], dim=1)
            if pocket_id == eot_mapped:
                break
    model.train()
    return instruction + '\n' + ''.join(tokens)

# --- Setup ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

ds = EfficientInstructDataset(context_length=CONTEXT_LEN)
config = AtomicConfig(vocab_size=VOCAB_SIZE, dim=DIM, depth=DEPTH, heads=HEADS, context_length=CONTEXT_LEN)
model = AtomicTransformer(config).to(device)

start_step = 0
ckpt_path = os.path.join(DRIVE_DIR, 'instruct_final.pt')

# Separate weight decay groups
decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if 'ln' in name or 'bias' in name or 'emb' in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = optim.AdamW([
    {'params': decay_params, 'weight_decay': WEIGHT_DECAY},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=LR)

# Resume from checkpoint
checkpoint = None
if os.path.exists(ckpt_path):
    print(f'>> Resuming from checkpoint: {ckpt_path}')
    checkpoint = torch.load(ckpt_path, map_location=device)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        if 'step' in checkpoint:
            start_step = checkpoint['step']
        if 'optimizer_state_dict' in checkpoint:
            try:
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                print('   Optimizer state restored.')
            except:
                print('   Warning: Could not restore optimizer state')
        if 'rng_state' in checkpoint:
            torch.set_rng_state(checkpoint['rng_state'])
            if 'np_rng_state' in checkpoint:
                np.random.set_state(checkpoint['np_rng_state'])
            print('   RNG state restored.')
    else:
        model.load_state_dict(checkpoint)
    print(f'   Resuming from step {start_step}')
else:
    print('>> Starting fresh Flagship model')
    init_weights(model)
    print('   Applied Kaiming initialization to BitLinear weights.')

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parameters: {total_params:.2f}M')

total_steps = start_step + ADDITIONAL_STEPS

# Cosine schedule with warmup
scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP_STEPS, total_steps)

if start_step > 0:
    for _ in range(start_step):
        scheduler.step()
    if checkpoint and 'scheduler_state_dict' in checkpoint:
        try:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            print('   Scheduler state restored.')
        except:
            print('   Warning: Could not restore scheduler, using re-computed state.')

scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and device == 'cuda'))
losses = []

# Training log on Drive
log_file = os.path.join(DRIVE_DIR, 'training_log.csv')
if not os.path.exists(log_file):
    with open(log_file, 'w') as f:
        f.write('step,loss,lr,step_time_ms\n')

print(f'Training from step {start_step} to {total_steps} (Grad Accum: {GRAD_ACCUM_STEPS}, Eff Batch: {BATCH_SIZE * GRAD_ACCUM_STEPS})...')
optimizer.zero_grad()
pbar = tqdm(range(start_step, total_steps), desc='Training')
step_start_time = time.time()

for step in pbar:
    # Gradient Accumulation
    loss_accum = 0.0
    for _ in range(GRAD_ACCUM_STEPS):
        x, y = ds.get_batch(BATCH_SIZE)
        x, y = x.to(device), y.to(device)

        with torch.amp.autocast('cuda', enabled=(USE_AMP and device == 'cuda')):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        loss_accum += loss.item()

    # Gradient clipping
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()
    scheduler.step()

    # Timing
    step_end_time = time.time()
    step_duration_ms = (step_end_time - step_start_time) * 1000
    step_start_time = step_end_time

    losses.append(loss_accum)
    current_lr = scheduler.get_last_lr()[0]
    pbar.set_postfix(loss=f'{loss_accum:.4f}', lr=f'{current_lr:.2e}', ms=f'{step_duration_ms:.0f}')

    # Log to CSV
    if step % 10 == 0:
        with open(log_file, 'a') as f:
            f.write(f'{step},{loss_accum:.5f},{current_lr:.5e},{step_duration_ms:.1f}\n')

    if step % 500 == 0 and step > 0:
        sample = generate_demo(model, ds, 'What is AI?')
        tqdm.write(f'\n--- Step {step} Sample ---\n{sample}\n')

    if step > 0 and step % 1000 == 0:
        save_dict = {
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'rng_state': torch.get_rng_state(),
            'np_rng_state': np.random.get_state(),
            'config': {
                'vocab_size': VOCAB_SIZE, 'dim': DIM, 'depth': DEPTH,
                'heads': HEADS, 'context_length': CONTEXT_LEN,
            },
        }
        torch.save(save_dict, ckpt_path)
        tqdm.write(f'💾 Checkpoint saved at step {step}')

# Final save
save_dict = {
    'step': total_steps,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'rng_state': torch.get_rng_state(),
    'np_rng_state': np.random.get_state(),
    'config': {
        'vocab_size': VOCAB_SIZE, 'dim': DIM, 'depth': DEPTH,
        'heads': HEADS, 'context_length': CONTEXT_LEN,
    },
}
torch.save(save_dict, ckpt_path)
print(f'\n✅ Training complete! Checkpoint saved to {ckpt_path}')

## 6 · Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve
axes[0].plot(losses, alpha=0.3, label='Raw')
window = min(100, len(losses) // 5) if len(losses) > 10 else 1
if window > 1:
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(losses)), smoothed, label=f'Smoothed ({window})')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LR schedule visualization
lr_schedule = []
temp_opt = optim.AdamW(model.parameters(), lr=LR)
temp_sched = get_cosine_schedule_with_warmup(temp_opt, WARMUP_STEPS, total_steps)
for _ in range(total_steps):
    lr_schedule.append(temp_sched.get_last_lr()[0])
    temp_sched.step()
axes[1].plot(lr_schedule)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('LR Schedule (Warmup + Cosine)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Atomic-1Bit Flagship Instruct (12.5M)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Generate samples
prompts = ['What is AI?', 'Explain gravity simply.', 'Write a haiku about computers.', 'Count to 5.']
print('\n📝 Generated Samples:')
print('=' * 60)
for p in prompts:
    sample = generate_demo(model, ds, p)
    print(f'\n{sample}')
    print('-' * 60)

## 7 · Download Checkpoint

Your checkpoint is already saved to Google Drive. To use it locally:

1. Go to [Google Drive](https://drive.google.com) → `Atomic-1Bit/weights/`
2. Download `instruct_final.pt`
3. Place it in your local `weights/` directory

Or download directly from Colab:

In [ ]:
# Optional: Download checkpoint directly from Colab
from google.colab import files
files.download(ckpt_path)